In [13]:
from openmm import app, unit, LangevinMiddleIntegrator

# Define water model
forcefield = app.ForceField('tip3p.xml')

# Create empty box
water_box_model = app.Modeller(topology=app.Topology(), positions=[])

print("ADD SOLVENT")
# Add water molecules to box
n_water_mols = 900
water_box_model.addSolvent(forcefield, numAdded=n_water_mols)

# Create system
water_box_system = forcefield.createSystem(
    water_box_model.topology, 
    nonbondedMethod=app.PME,
    nonbondedCutoff=1*unit.nanometer, 
    constraints=app.HBonds,
)

# Define integrator
integrator = LangevinMiddleIntegrator(300 * unit.kelvin, 1/unit.picosecond, 0.001*unit.picoseconds)

print(water_box_model.topology)
print("MINIMISING")
# Construct simulation and minimise energy
simulation = app.Simulation(water_box_model.topology, water_box_system, integrator)
simulation.context.setPositions(water_box_model.positions)
simulation.minimizeEnergy()

FILENAME = f"solvent-{n_water_mols*3}.xml"
print("WRITING", FILENAME)
from nanover.openmm.serializer import serialize_simulation

with open(FILENAME, "w") as outfile:
    outfile.write(serialize_simulation(simulation))

print("DONE")

ADD SOLVENT
<Topology; 1 chains, 900 residues, 2700 atoms, 1800 bonds>
MINIMISING
WRITING solvent-2700.xml
DONE


In [2]:
from nanover.openmm import OpenMMSimulation

omm = OpenMMSimulation.from_xml_path("solvent-300000.xml")
omm.load()
frame = omm.make_topology_frame()
print(frame, "\n", frame.box_vectors)

<FrameData of bond.pairs[200000, 2], chain.count, chain.names[1], energy.kinetic, energy.potential, particle.count, particle.elements[300000], particle.names[300000], particle.positions[300000, 3], particle.residues[300000], residue.chains[100000], residue.count, residue.ids[100000], residue.names[100000], system.box.vectors[3, 3], system.simulation.time> 
 [[14.5484  0.      0.    ]
 [ 0.     14.5484  0.    ]
 [ 0.      0.     14.5484]]


In [3]:
from contextlib import suppress
from nanover.omni import OmniRunner

with suppress(NameError):
    runner.close()

runner = OmniRunner.with_basic_server(omm, name="PERF TEST", port=0)
runner.load(0)

def set_shared_value(key, value):
    from nanover.utilities.change_buffers import DictionaryChange
    updates = {key: value}
    change = DictionaryChange(updates=updates)
    runner.app_server.update_state(change)

set_shared_value("suggested.report.connection.health", True)